# 0. Load imports 

In [123]:
import pandas as pd
import numpy as np
import re
import gdown #needed to get data from google drive

import matplotlib.pyplot as plt
import seaborn as sns

## print multiple things from same cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## load data of post 2000 UN voting data
gdown.download(id="1aHnVYItfvHossY7LpmnsQimvx1wNko1f", output="2026_02_06_ga_voting.csv", quiet=False)
df = pd.read_csv("2026_02_06_ga_voting.csv")

print(df.shape)
print(df.head())


Downloading...
From (original): https://drive.google.com/uc?id=1aHnVYItfvHossY7LpmnsQimvx1wNko1f
From (redirected): https://drive.google.com/uc?id=1aHnVYItfvHossY7LpmnsQimvx1wNko1f&confirm=t&uuid=e0beb65b-99de-49ec-acda-9508f0c77549
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/2026_02_06_ga_voting.csv
100%|████████████████████████████████████████| 364M/364M [00:17<00:00, 21.2MB/s]


'2026_02_06_ga_voting.csv'

/var/folders/cs/x74m_3cj4xv9_rl_f_d37ts40000gn/T/ipykernel_38414/2184471.py:15: DtypeWarning: Columns (5,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("2026_02_06_ga_voting.csv")


(947434, 20)
   undl_id ms_code      ms_name ms_vote        date session   resolution  \
0   507407     AFG  AFGHANISTAN       Y  2003-12-03      58  A/RES/58/20   
1   507407     ALB      ALBANIA       Y  2003-12-03      58  A/RES/58/20   
2   507407     DZA      ALGERIA       Y  2003-12-03      58  A/RES/58/20   
3   507407     AND      ANDORRA       Y  2003-12-03      58  A/RES/58/20   
4   507407     AGO       ANGOLA       X  2003-12-03      58  A/RES/58/20   

                       draft committee_report     meeting  \
0  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
1  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
2  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
3  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   
4  A/58/L.25|A/58/L.25/Add.1              NaN  A/58/PV.68   

                                               title            agenda_title  \
0  Special information programme on the question ...  Question of Palestine.   
1  

# 1. Filter data to post 2000

I considered doing this by session, but sessions run Sept-Sept, so it would be harder to isolate Trump's changes.

In [124]:
## Filter to post 2000
df['year'] = pd.to_datetime(df['date']).dt.year
df_post2000 = df[df['year'] >= 2000].copy()

# 2. Exploring the data

In [125]:
## How many votes per year
print(df_post2000['year'].min())
print(df_post2000['year'].value_counts().sort_index())

2000
year
2000    12851
2001    12852
2002    14508
2003    14707
2004    13943
2005    14325
2006    19390
2007    15168
2008    14976
2009    13056
2010    13824
2011    13312
2012    14282
2013    12352
2014    15440
2015    15054
2016    15633
2017    18142
2018    20651
2019    19300
2020    19300
2021    16598
2022    17177
2023    16984
2024    18335
2025    37056
Name: count, dtype: int64


In [5]:
## How many resolutions per year and total number of resolutions
df_post2000.groupby('year')['resolution'].nunique().sort_index()
print(df_post2000['resolution'].nunique())

year
2000     68
2001     68
2002     76
2003     77
2004     73
2005     75
2006    101
2007     79
2008     78
2009     68
2010     72
2011     69
2012     74
2013     64
2014     80
2015     78
2016     81
2017     94
2018    107
2019    100
2020    100
2021     86
2022     89
2023     88
2024     95
2025    192
Name: resolution, dtype: int64

2232


In [24]:
## Number of unique countries
print(df_post2000['ms_name'].value_counts())
print(df_post2000['ms_name'].nunique())

ms_name
AFGHANISTAN                     2232
MYANMAR                         2232
NAURU                           2232
NEPAL                           2232
NEW ZEALAND                     2232
                                ... 
NETHERLANDS (KINGDOM OF THE)     365
VENEZUELA                        297
SERBIA AND MONTENEGRO            227
YUGOSLAVIA                       212
TÜRKÝYE                           70
Name: count, Length: 206, dtype: int64
206


In [6]:
## What are the voting options
print(df_post2000['ms_vote'].unique())
print(df_post2000['ms_vote'].value_counts())

['Y' 'X' 'A' 'N']
ms_vote
Y    318454
A     44186
X     39503
N     27073
Name: count, dtype: int64


In [7]:
## are there any NAs?
print(df_post2000['ms_vote'].isna().sum())

0


# 3. Cleaning the country titles to resolve differing names

I noticed how some countries had multiple names (e.g. Turkey). However, they have the same country code. So I standardized the country names so that each code has just 1. 

In [126]:
print(sorted(df_post2000['ms_name'].unique()))

['AFGHANISTAN', 'ALBANIA', 'ALGERIA', 'ANDORRA', 'ANGOLA', 'ANTIGUA AND BARBUDA', 'ARGENTINA', 'ARMENIA', 'AUSTRALIA', 'AUSTRIA', 'AZERBAIJAN', 'BAHAMAS', 'BAHRAIN', 'BANGLADESH', 'BARBADOS', 'BELARUS', 'BELGIUM', 'BELIZE', 'BENIN', 'BHUTAN', 'BOLIVIA', 'BOLIVIA (PLURINATIONAL STATE OF)', 'BOSNIA AND HERZEGOVINA', 'BOTSWANA', 'BRAZIL', 'BRUNEI DARUSSALAM', 'BULGARIA', 'BURKINA FASO', 'BURUNDI', 'CABO VERDE', 'CAMBODIA', 'CAMEROON', 'CANADA', 'CAPE VERDE', 'CENTRAL AFRICAN REPUBLIC', 'CHAD', 'CHILE', 'CHINA', 'COLOMBIA', 'COMOROS', 'CONGO', 'COSTA RICA', "COTE D'IVOIRE", 'CROATIA', 'CUBA', 'CYPRUS', 'CZECH REPUBLIC', 'CZECHIA', "CÔTE D'IVOIRE", "DEMOCRATIC PEOPLE'S REPUBLIC OF KOREA", 'DEMOCRATIC REPUBLIC OF THE CONGO', 'DENMARK', 'DJIBOUTI', 'DOMINICA', 'DOMINICAN REPUBLIC', 'ECUADOR', 'EGYPT', 'EL SALVADOR', 'EQUATORIAL GUINEA', 'ERITREA', 'ESTONIA', 'ESWATINI', 'ETHIOPIA', 'FIJI', 'FINLAND', 'FRANCE', 'GABON', 'GAMBIA', 'GEORGIA', 'GERMANY', 'GHANA', 'GREECE', 'GRENADA', 'GUATEMALA',

In [127]:
# for each country code, what country names correspond to them. filters to those which have more than 1.
# output is the code and the multiple names associated w that code

code_name_check = df_post2000.groupby('ms_code')['ms_name'].nunique()
problem_codes = code_name_check[code_name_check > 1].index
for code in problem_codes:
    print(code, df_post2000[df_post2000['ms_code'] == code]['ms_name'].unique())

BOL ['BOLIVIA' 'BOLIVIA (PLURINATIONAL STATE OF)']
CIV ["CÔTE D'IVOIRE" "COTE D'IVOIRE"]
CPV ['CAPE VERDE' 'CABO VERDE']
CZE ['CZECH REPUBLIC' 'CZECHIA']
LBY ['LIBYAN ARAB JAMAHIRIYA' 'LIBYA']
MKD ['THE FORMER YUGOSLAV REPUBLIC OF MACEDONIA' 'NORTH MACEDONIA']
NLD ['NETHERLANDS' 'NETHERLANDS (KINGDOM OF THE)']
SWZ ['SWAZILAND' 'ESWATINI']
TUR ['TURKEY' 'TÜRKİYE' 'TÜRKÝYE']
VEN ['VENEZUELA' 'VENEZUELA (BOLIVARIAN REPUBLIC OF)']


In [128]:
## standardize the names
code_to_name = {
    'BOL': 'BOLIVIA',
    'CIV': "COTE D'IVOIRE",
    'CPV': 'CABO VERDE',
    'CZE': 'CZECHIA',
    'LBY': 'LIBYA',
    'MKD': 'NORTH MACEDONIA',
    'NLD': 'NETHERLANDS',
    'SWZ': 'ESWATINI',
    'TUR': 'TURKEY',
    'VEN': 'VENEZUELA'
}
df_post2000['ms_name'] = df_post2000['ms_code'].map(code_to_name).fillna(df_post2000['ms_name'])

print(df_post2000['ms_name'].nunique())

195


# 4. Cleaning the voting options

I found that there are 4 voting options: Yes, No, Abstain and Absent. I decided to remove absent, but usually abstaining has meaning . Therefore, I will keep that in as a middle ground value of 0.5, while No is 0 and Yes is 1.

In [129]:
df_post2000 = df_post2000[df_post2000['ms_vote'] != 'X']
print(df_post2000['ms_vote'].value_counts())

ms_vote
Y    318454
A     44186
N     27073
Name: count, dtype: int64


In [131]:
vote_score = {'Y': 1, 'A': 0.5, 'N': 0}
df_post2000['ms_vote_score'] = df_post2000['ms_vote'].map(vote_score)

# 5. Sorting the Resolutions into specific topics

Llama does zero-shot classification whereby it sees the resolution title, and assigns the resolution a topic category (LLM). It isn't perfect, so I then clean up the titles. Then, I merge them onto the main dataset.

In [132]:
# sample of resolution titles

titles = df_post2000[['resolution', 'title']].drop_duplicates()
print(titles['title'].sample(50).tolist())

['The right to food : resolution / adopted by the General Assembly', 'Universal Declaration on the Achievement of a Nuclear-Weapon-Free World : resolution / adopted by the General Assembly', 'Necessity of ending the economic, commercial and financial embargo imposed by the United States of America against Cuba : resolution / adopted by the General Assembly', 'Implementation of the Convention on Cluster Munitions : resolution / adopted by the General Assembly', 'The right to development : resolution / adopted by the General Assembly', 'Promotion of inclusive and effective international tax cooperation at the United Nations : resolution / adopted by the General Assembly', 'Implementation of the Declaration on the Granting of Independence to Colonial Countries and Peoples by the specialized agencies and the international institutions associated with the United Nations : resolution / adopted by the General Assembly', 'Implementation of the Convention on the Prohibition of the Use, Stockpil

In [133]:
## provides the different categories I want in the prompt.

import requests

prompt = """
Classify this UN resolution title into exactly one of these categories:
Palestine and the Middle East, Nuclear Disarmament, Development and Aid, 
Environment, Armed Conflict & Security, Human Rights, Decolonization and Self Determination, 
Economic and Trade, UN Institutional, Other.

Respond with ONLY the category name, nothing else.
"""

def classify_with_llama(title):
    response = requests.post('http://localhost:11434/api/generate',
        json={
            "model": "llama3",
            "prompt": f"{prompt}\n\nTitle: {title}",
            "stream": False
        })
    return response.json()['response'].strip()



In [ ]:
#titles['category'] = titles['title'].apply(classify_with_llama)

In [134]:
from tqdm import tqdm
tqdm.pandas()

titles['category'] = titles['title'].progress_apply(classify_with_llama) #this allows me to see how long it is taking

100%|███████████████████████████████████████| 2232/2232 [15:18<00:00,  2.43it/s]


In [17]:
print(titles['category'].value_counts())

category
Armed Conflict & Security                531
Human Rights                             404
Nuclear Disarmament                      400
Palestine and the Middle East            271
Decolonization and Self Determination    221
Development and Aid                      182
Environment                               77
UN Institutional                          75
Economic and Trade                        61
Environmental                              6
Other                                      2
Other.                                     1
Health                                     1
Name: count, dtype: int64


In [136]:
## clean up the messy titles

replacements = {
    'Environmental': 'Environment',
    'International Law': 'Other',
    'Information': 'Other',
    'Other.': 'Other',
    'Health': 'Other'
}
titles['category'] = titles['category'].replace(replacements)
print(titles['category'].value_counts())

category
Armed Conflict & Security                529
Human Rights                             397
Nuclear Disarmament                      396
Palestine and the Middle East            268
Decolonization and Self Determination    225
Development and Aid                      183
Environment                               92
UN Institutional                          75
Economic and Trade                        62
Other                                      5
Name: count, dtype: int64


In [137]:
#merging

df_post2000 = df_post2000.merge(titles[['resolution', 'category']], on='resolution', how='left')

print(df_post2000['category'].isna().sum()) 

0


# 6. Alignment Scores for the US and China

In [138]:
# Isolating what the US and China voted in all these resolutions, and adding that as a column to all countries and their votes
us_votes = df_post2000[df_post2000['ms_code'] == 'USA'][['resolution', 'ms_vote_score']].rename(columns={'ms_vote_score': 'us_vote_score'})
china_votes = df_post2000[df_post2000['ms_code'] == 'CHN'][['resolution', 'ms_vote_score']].rename(columns={'ms_vote_score': 'china_vote_score'})

#merge onto full dataset 
df_post2000 = df_post2000.merge(us_votes, on='resolution', how='left')
df_post2000 = df_post2000.merge(china_votes, on='resolution', how='left')

# 7. Filter to votes where the US and China are misaligned

In [139]:
df_post2000 = df_post2000[df_post2000['us_vote_score'] != df_post2000['china_vote_score']]
df_post2000

,undl_id,ms_code,ms_name,ms_vote,date,session,resolution,draft,committee_report,meeting,...,total_no,total_abstentions,total_non_voting,total_ms,undl_link,year,ms_vote_score,category,us_vote_score,china_vote_score
0,507407,AFG,AFGHANISTAN,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0
1,507407,ALB,ALBANIA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0
2,507407,DZA,ALGERIA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0
3,507407,AND,ANDORRA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0
4,507407,ATG,ANTIGUA AND BARBUDA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,6.0,6.0,20.0,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389708,4088513,URY,URUGUAY,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0
389709,4088513,UZB,UZBEKISTAN,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0
389710,4088513,VNM,VIET NAM,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0
389711,4088513,YEM,YEMEN,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,5.0,6.0,37.0,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0


# 8. For each year, alignment score for each country with the US/China

Perfect alignment = 1. To calculate the alignment w US/China, take the absolute value of the difference between their score and US score and remove it from 1.

In [140]:
df_post2000['align_us'] = 1 - abs(df_post2000['ms_vote_score'] - df_post2000['us_vote_score'])
df_post2000['align_china'] = 1 - abs(df_post2000['ms_vote_score'] - df_post2000['china_vote_score'])

# 9. Load and Merge in USAID data

Now I merge in the foreign assistance data. First, I explored how much country code overlap we had. We had 189/195, with countries in UN, but not in aid are: US (as doesn't receive own aid), 4 microstates in Europe and Yugoslavia. I filtered to just disbursements as this is money that has been transfered, while other categories haven't yet.

In [141]:
## now adding in USAID
gdown.download(id="1olxIzOBAbjpbLfrlvFkPqAfDF2UFe-UI", output="us_foreign_aid_country.csv", quiet=False)
aid = pd.read_csv("us_foreign_aid_country.csv")

print(aid.shape)
print(aid.columns.tolist())
print(aid.head())

Downloading...
From: https://drive.google.com/uc?id=1olxIzOBAbjpbLfrlvFkPqAfDF2UFe-UI
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/us_foreign_aid_country.csv
100%|██████████████████████████████████████| 2.41M/2.41M [00:00<00:00, 19.5MB/s]


'us_foreign_aid_country.csv'

(22763, 11)
['Country Code', 'Country Name', 'Region ID', 'Region Name', 'Income Group Acronym', 'Income Group Name', 'Transaction Type ID', 'Transaction Type Name', 'Fiscal Year', 'current_amount', 'constant_amount']
  Country Code Country Name  Region ID         Region Name  \
0          ABW        Aruba          6  Western Hemisphere   
1          ABW        Aruba          6  Western Hemisphere   
2          ABW        Aruba          6  Western Hemisphere   
3          ABW        Aruba          6  Western Hemisphere   
4          ABW        Aruba          6  Western Hemisphere   

  Income Group Acronym    Income Group Name  Transaction Type ID  \
0                  HIC  High Income Country                    2   
1                  HIC  High Income Country                    2   
2                  HIC  High Income Country                    2   
3                  HIC  High Income Country                    2   
4                  HIC  High Income Country                    2   



In [142]:
voting_codes = set(df_post2000['ms_code'].unique())
aid_codes = set(aid['Country Code'].unique())

print(f"Voting codes: {len(voting_codes)}")
print(f"Aid codes: {len(aid_codes)}")
print(f"Overlap: {len(voting_codes.intersection(aid_codes))}")
print(f"\nIn voting but not in aid:")
print(voting_codes - aid_codes)
print(f"\nIn aid but not in voting:")
print(aid_codes - voting_codes)

Voting codes: 195
Aid codes: 249
Overlap: 189

In voting but not in aid:
{'USA', 'SMR', 'AND', 'YUG', 'LIE', 'MCO'}

In aid but not in voting:
{'COK', 'YUF', 'MSR', 'PIT', 'NCN', 'WHN', 'PYF', 'AIA', 'SDF', 'MTQ', 'SSN', 'HKG', 'ABW', 'CCN', 'MNS', 'ANS', 'CRN', 'CYM', 'EAG', 'CUW', 'CS-KM', 'WEC', 'EAQ', 'LAN', 'TWN', 'EEE', 'CNA', 'IOT', 'BMU', 'TIB', 'AFR', 'MAC', 'ASN', 'LCN', 'SAG', 'MNA', 'SCN', 'EES', 'TCA', nan, 'WLD', 'SAN', 'OCN', 'EEN', 'ANT', 'ECN', 'EUS', 'VGB', 'EUN', 'PSE', 'ESN', 'NAN', 'GUF', 'SEA', 'WAN', 'MEN', 'GRL', 'NCL', 'ENS', 'SMN'}


In [143]:
print(aid['Transaction Type Name'].value_counts())

Transaction Type Name
Obligations                    11947
Disbursements                   5500
Appropriated and Planned        2742
President's Budget Requests     2574
Name: count, dtype: int64


In [144]:
## filter to just disbursements
aid_clean = aid[aid['Transaction Type Name'] == 'Disbursements'].copy()

In [145]:
## sum for each country and year the amount of disbursements they've received
aid_agg = aid_clean.groupby(['Country Code', 'Fiscal Year'])['constant_amount'].sum().reset_index()
aid_agg.columns = ['ms_code', 'year', 'usaid_disbursement']

In [146]:
## now merge onto our voting data
df_post2000['year'] = df_post2000['year'].astype(int)
aid_agg['year'] = aid_agg['year'].astype(int)

df_post2000 = df_post2000.merge(aid_agg, on=['ms_code', 'year'], how='left')

print(df_post2000['usaid_disbursement'].isna().sum())
print(df_post2000.shape)

df_post2000

40670
(346250, 28)


,undl_id,ms_code,ms_name,ms_vote,date,session,resolution,draft,committee_report,meeting,...,total_ms,undl_link,year,ms_vote_score,category,us_vote_score,china_vote_score,align_us,align_china,usaid_disbursement
0,507407,AFG,AFGHANISTAN,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,1.152245e+09
1,507407,ALB,ALBANIA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,8.570566e+07
2,507407,DZA,ALGERIA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,6.003427e+06
3,507407,AND,ANDORRA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,NaN
4,507407,ATG,ANTIGUA AND BARBUDA,Y,2003-12-03,58,A/RES/58/20,A/58/L.25|A/58/L.25/Add.1,NaN,A/58/PV.68,...,191.0,https://digitallibrary.un.org/record/507407,2003,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,5.393330e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346245,4088513,URY,URUGUAY,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,4.912320e+05
346246,4088513,UZB,UZBEKISTAN,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,4.216949e+07
346247,4088513,VNM,VIET NAM,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,1.389279e+08
346248,4088513,YEM,YEMEN,Y,2025-09-19,80,A/RES/80/1,A/80/L.2/Rev.1,NaN,A/80/PV.3,...,193.0,https://digitallibrary.un.org/record/4088513,2025,1.0,Palestine and the Middle East,0.0,1.0,0.0,1.0,2.634188e+08


In [156]:
aid_clean.groupby('Fiscal Year')['constant_amount'].sum().sort_index()

Fiscal Year
2001    25241607994
2002    30871894657
2003    38458126032
2004    41387918523
2005    50466673471
2006    48652539535
2007    50475930726
2008    59940193773
2009    62934476565
2010    58531508075
2011    64764469232
2012    62061221822
2013    60850840278
2014    54163674816
2015    62040081981
2016    59529896554
2017    56814062272
2018    57338180554
2019    56109053697
2020    56027540718
2021    56660607342
2022    79660860493
2023    82471907808
2024    71609065797
2025    39189955589
2026     2684416281
Name: constant_amount, dtype: int64

# 10. Ensuring USAID is a % of GDP

I collected World Bank GDP data in current $. For years where data is missing (predominantly 2025), I used the most recent year's data e.g. for 2025, did 2024.

In [148]:
## getting GDP data
gdown.download(id="103NZ00N4vczRyM6JASR1aWu3Osws91vI", output="worldbank_GDP_data.csv", quiet=False)

gdp_raw = pd.read_csv("worldbank_GDP_data.csv", skiprows = 4)

print(gdp.head())

Downloading...
From: https://drive.google.com/uc?id=103NZ00N4vczRyM6JASR1aWu3Osws91vI
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/worldbank_GDP_data.csv
100%|████████████████████████████████████████| 187k/187k [00:00<00:00, 1.16MB/s]


'worldbank_GDP_data.csv'

                  Country Name Country Code  year           gdp
0                        ARUBA          ABW  2000  1.873453e+09
1  AFRICA EASTERN AND SOUTHERN          AFE  2000  2.870420e+11
2                  AFGHANISTAN          AFG  2000  3.521418e+09
3   AFRICA WESTERN AND CENTRAL          AFW  2000  1.427000e+11
4                       ANGOLA          AGO  2000  9.129595e+09


In [149]:
gdp = gdp_raw[['Country Name', 'Country Code'] + [str(y) for y in range(2000, 2026)]].copy()

# forward fill missing years within each country (does most recent year available)
gdp = gdp.set_index(['Country Name', 'Country Code'])
gdp = gdp.ffill(axis=1) 
gdp = gdp.reset_index()

# reshape to long format to make it mergable
gdp = gdp.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='year',
    value_name='gdp'
)

gdp['year'] = gdp['year'].astype(int)
gdp['Country Name'] = gdp['Country Name'].str.upper()
gdp = gdp.dropna(subset=['gdp'])

# carry 2024 forward to 2025 for data as no 2025 data yet
gdp_2024 = gdp[gdp['year'] == 2024].copy()
gdp_2024['year'] = 2025
gdp = pd.concat([gdp, gdp_2024], ignore_index=True)

print(gdp.shape)
print(gdp.head())

(7023, 4)
                  Country Name Country Code  year           gdp
0                        ARUBA          ABW  2000  1.873453e+09
1  AFRICA EASTERN AND SOUTHERN          AFE  2000  2.870420e+11
2                  AFGHANISTAN          AFG  2000  3.521418e+09
3   AFRICA WESTERN AND CENTRAL          AFW  2000  1.427000e+11
4                       ANGOLA          AGO  2000  9.129595e+09


In [150]:
## before merging onto the main dataset, how much overlap is there?

voting_names = set(df_post2000['ms_name'].unique())
gdp_names = set(gdp['Country Name'].unique())

print(f"Overlap: {len(voting_names.intersection(gdp_names))}")
print("Not matching:", voting_names - gdp_names)

Overlap: 172
Not matching: {'UNITED REPUBLIC OF TANZANIA', 'VENEZUELA', 'SLOVAKIA', 'EGYPT', 'SOMALIA', 'DEMOCRATIC REPUBLIC OF THE CONGO', 'REPUBLIC OF KOREA', 'KYRGYZSTAN', 'YEMEN', 'SAINT KITTS AND NEVIS', 'REPUBLIC OF MOLDOVA', 'IRAN (ISLAMIC REPUBLIC OF)', 'MICRONESIA (FEDERATED STATES OF)', 'YUGOSLAVIA', 'CONGO', 'SERBIA AND MONTENEGRO', 'BAHAMAS', 'GAMBIA', 'SAINT VINCENT AND THE GRENADINES', 'SAINT LUCIA', "LAO PEOPLE'S DEMOCRATIC REPUBLIC", 'TURKEY', "DEMOCRATIC PEOPLE'S REPUBLIC OF KOREA"}


In [151]:
## map names to make merge possible
gdp_name_mapping = {
    'UNITED REPUBLIC OF TANZANIA': 'TANZANIA',
    'VENEZUELA': 'VENEZUELA, RB',
    'SLOVAKIA': 'SLOVAK REPUBLIC',
    'EGYPT': 'EGYPT, ARAB REP.',
    'DEMOCRATIC REPUBLIC OF THE CONGO': 'CONGO, DEM. REP.',
    'REPUBLIC OF KOREA': 'KOREA, REP.',
    'KYRGYZSTAN': 'KYRGYZ REPUBLIC',
    'YEMEN': 'YEMEN, REP.',
    'SAINT KITTS AND NEVIS': 'ST. KITTS AND NEVIS',
    'REPUBLIC OF MOLDOVA': 'MOLDOVA',
    'IRAN (ISLAMIC REPUBLIC OF)': 'IRAN, ISLAMIC REP.',
    'MICRONESIA (FEDERATED STATES OF)': 'MICRONESIA, FED. STS.',
    'CONGO': 'CONGO, REP.',
    'BAHAMAS': 'BAHAMAS, THE',
    'GAMBIA': 'GAMBIA, THE',
    'SAINT VINCENT AND THE GRENADINES': 'ST. VINCENT AND THE GRENADINES',
    'SAINT LUCIA': 'ST. LUCIA',
    
    "LAO PEOPLE'S DEMOCRATIC REPUBLIC": 'LAO PDR',
    'TURKEY': 'TURKIYE',
    "DEMOCRATIC PEOPLE'S REPUBLIC OF KOREA": "KOREA, DEM. PEOPLE'S REP.",
    'SOMALIA': 'SOMALIA, FED. REP.'
}

df_post2000['country_name_gdp'] = df_post2000['ms_name'].replace(gdp_name_mapping)
print(df_post2000['country_name_gdp'].sample(10).tolist())

gdp_name_mapping['SOMALIA'] = 'SOMALIA, FED. REP.'

df_post2000['country_name_gdp'] = df_post2000['ms_name'].replace(gdp_name_mapping)

['LITHUANIA', 'VENEZUELA, RB', 'SWEDEN', 'BOLIVIA', 'TURKIYE', 'MOZAMBIQUE', 'ARMENIA', 'SOUTH SUDAN', 'LEBANON', 'ISRAEL']


In [152]:
## now merge!!
alignment_aid_gdp = df_post2000.merge(
    gdp[['Country Name', 'year', 'gdp']],
    left_on=['country_name_gdp', 'year'],
    right_on=['Country Name', 'year'],
    how='left'
).drop(columns=['Country Name', 'country_name_gdp'])

print(alignment_aid_gdp['gdp'].isna().sum())
print(alignment_aid_gdp.shape)


2212
(378081, 29)


In [153]:
## very few NAs (<1%) --> check what they are 
na_countries = alignment_aid_gdp[alignment_aid_gdp['gdp'].isna()]['ms_name'].unique()
print(sorted(na_countries))

## north korea and 2 countries that no longer exist

["DEMOCRATIC PEOPLE'S REPUBLIC OF KOREA", 'SERBIA AND MONTENEGRO', 'YUGOSLAVIA']


In [154]:
## now make a new column of aid % of gdp
alignment_aid_gdp['aid_gdp_ratio'] = (alignment_aid_gdp['usaid_disbursement'] / alignment_aid_gdp['gdp']) * 100

# sanity check
print(alignment_aid_gdp[['ms_name', 'year', 'usaid_disbursement', 'gdp', 'aid_gdp_ratio']].dropna().head(10))

                ms_name  year  usaid_disbursement           gdp  aid_gdp_ratio
0           AFGHANISTAN  2003        1.152245e+09  4.520947e+09      25.486810
1               ALBANIA  2003        8.570566e+07  5.801712e+09       1.477248
2               ALGERIA  2003        6.003427e+06  7.348226e+10       0.008170
4   ANTIGUA AND BARBUDA  2003        5.393330e+05  9.481000e+08       0.056886
5             ARGENTINA  2003        9.127823e+06  1.275870e+11       0.007154
6               ARMENIA  2003        1.552192e+08  2.807061e+09       5.529598
9            AZERBAIJAN  2003        1.016739e+08  7.276413e+09       1.397308
10              BAHAMAS  2003        1.016494e+07  8.870090e+09       0.114598
11              BAHRAIN  2003        1.535935e+08  1.107481e+10       1.386872
12           BANGLADESH  2003        1.516008e+08  6.015893e+10       0.252001


# 11. Save this cleaned data

I saved it, but was too large for Github, so i dragged it into the Google Drive. 

In [155]:
alignment_aid_gdp.to_csv('../data/processed/df_clean.csv', index=False)